# 02 - Data Cleaning

Compiled version of the cleaning notebook.

Design choices:
- Avoid forward-filling: it creates artificial zero returns on cross-market holidays, which contaminate the signal.
- Log returns for price assets (Gold, Brent, DXY); level changes for VIX and US10Y.
- Strict dropna alignment across all five core assets.
- Save core aligned prices and modelling variables for downstream notebooks.

## Reader Orientation

This notebook protects the signal from data-quality mistakes. Since the dashboard will flag abnormal market behaviour, calendar mismatches or artificial forward-filled returns could create false warnings. The core output is a strictly aligned dataset for the main Brent-based nowcasting test.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs" / "step02"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
CORE_ASSETS = ["Gold", "Brent", "DXY", "VIX", "US10Y"]
CORE_PRICE_ASSETS = ["Gold", "Brent", "DXY"]
LEVEL_ASSETS = ["VIX", "US10Y"]

prices = pd.read_parquet(RAW_DIR / "prices_close_raw.parquet")
prices = prices[CORE_ASSETS].sort_index()

print("Raw close-price panel:", prices.shape)
prices.tail()

Raw close-price panel: (4768, 5)


Ticker,Gold,Brent,DXY,VIX,US10Y
Date,,,,,
2026-05-29,4560.500000,92.050003,98.910004,15.320000,4.453
2026-06-01,4475.200195,94.980003,99.199997,16.049999,4.475
2026-06-02,4489.100098,96.000000,99.220001,15.770000,4.455
2026-06-03,4436.700195,97.809998,99.529999,16.059999,4.491
2026-06-04,4502.299805,95.019997,99.349998,15.690000,4.465


In [3]:
missing_audit = pd.DataFrame({
    "missing_count": prices.isna().sum(),
    "missing_pct": prices.isna().mean().mul(100).round(3),
})

rows_with_missing = prices.loc[prices.isna().any(axis=1)].copy()

print("Rows with at least one missing value:", len(rows_with_missing))
missing_audit

Rows with at least one missing value: 82


,missing_count,missing_pct
Ticker,,
Gold,7,0.147
Brent,77,1.615
DXY,5,0.105
VIX,5,0.105
US10Y,8,0.168


### Why Audit Missing Values First

The signal is based on abnormal returns and changing correlations, so data alignment errors can become false alarms. The latest missing-value audit shows all core assets have less than 1% missing data, with Brent the highest at 37 rows. That is small enough to use a strict aligned sample without materially damaging the dataset.

For the modelling sample we use strict alignment across all five core assets. Any row where any asset is missing is dropped entirely. This avoids forward-filling, which would produce artificial zero returns on calendar mismatch days and contaminate volatility and z-score estimates.

In [4]:
core_prices = prices[CORE_ASSETS].dropna().copy()

core_log_returns = np.log(core_prices[CORE_PRICE_ASSETS] / core_prices[CORE_PRICE_ASSETS].shift(1))
core_level_changes = core_prices[LEVEL_ASSETS].diff()

market_vars = pd.concat(
    [
        core_log_returns.rename(columns={asset: f"r_{asset}" for asset in CORE_PRICE_ASSETS}),
        core_level_changes.rename(columns={asset: f"d_{asset}" for asset in LEVEL_ASSETS}),
    ],
    axis=1,
).dropna()

print("Core aligned prices:", core_prices.shape)
print("Core market variables:", market_vars.shape)
market_vars.tail()


Core aligned prices: (4686, 5)
Core market variables: (4685, 5)


Ticker,r_Gold,r_Brent,r_DXY,d_VIX,d_US10Y
Date,,,,,
2026-05-29,0.013510,-0.017873,-0.001111,-0.420000,-0.002
2026-06-01,-0.018881,0.031334,0.002928,0.730000,0.022
2026-06-02,0.003101,0.010682,0.000202,-0.279999,-0.020
2026-06-03,-0.011741,0.018679,0.003119,0.289999,0.036
2026-06-04,0.014677,-0.028939,-0.001810,-0.370000,-0.026


### Result Comment And Significance

Log returns are used for Gold, Brent, and DXY because they are price-level assets; VIX and US10Y use level changes because they are already index and yield levels where percentage changes are less meaningful. The strict alignment drops a small number of rows — expected given cross-market calendar differences — and leaves a clean, contiguous daily dataset for signal construction.

In [5]:
extreme_rows = []
for col in market_vars.columns:
    z = (market_vars[col] - market_vars[col].mean()) / market_vars[col].std()
    flagged = market_vars.loc[z.abs() > 5, [col]].copy()
    flagged["z_score"] = z.loc[flagged.index]
    flagged["variable"] = col
    extreme_rows.append(flagged.reset_index(names="date"))

extreme_moves = pd.concat(extreme_rows, ignore_index=True) if extreme_rows else pd.DataFrame()
extreme_moves = extreme_moves.sort_values(["variable", "date"]) if not extreme_moves.empty else extreme_moves

print("Extreme move rows:", len(extreme_moves))
extreme_moves.head(30)

Extreme move rows: 63


Ticker,date,r_Gold,z_score,variable,r_Brent,r_DXY,d_VIX,d_US10Y
58,2008-09-19,NaN,5.660396,d_US10Y,NaN,NaN,NaN,0.332
59,2008-12-17,NaN,-5.845430,d_US10Y,NaN,NaN,NaN,-0.343
60,2009-03-18,NaN,-8.010232,d_US10Y,NaN,NaN,NaN,-0.470
61,2009-05-01,NaN,6.614953,d_US10Y,NaN,NaN,NaN,0.388
62,2022-11-10,NaN,-5.487473,d_US10Y,NaN,NaN,NaN,-0.322
29,2008-09-29,NaN,5.960251,d_VIX,NaN,NaN,11.980000,NaN
30,2008-10-13,NaN,-7.441612,d_VIX,NaN,NaN,-14.959995,NaN
31,2008-10-15,NaN,7.024838,d_VIX,NaN,NaN,14.119999,NaN
32,2008-10-20,NaN,-8.635545,d_VIX,NaN,NaN,-17.360001,NaN
33,2008-10-22,NaN,8.228719,d_VIX,NaN,NaN,16.540001,NaN


### Result Comment And Significance

The extreme-move audit should not be treated as an automatic cleaning list. Large VIX, yield, Brent, or gold moves are often exactly the stress events the project wants to study. The purpose is to identify obvious data errors; if the extreme rows correspond to real market events, they should generally stay in the sample.

In [6]:
core_prices.to_parquet(PROCESSED_DIR / "prices_clean_core.parquet")
market_vars.to_parquet(PROCESSED_DIR / "market_vars_core.parquet")

# Compatibility aliases used by downstream notebooks.
core_prices.to_parquet(PROCESSED_DIR / "prices_clean.parquet")
market_vars.to_parquet(PROCESSED_DIR / "market_vars.parquet")

missing_audit.to_csv(OUTPUT_DIR / "missing_audit.csv")
rows_with_missing.to_csv(OUTPUT_DIR / "rows_with_missing.csv")
extreme_moves.to_csv(OUTPUT_DIR / "extreme_move_audit.csv", index=False)

print("Saved Step 02 outputs.")
print("Date range:", market_vars.index.min(), "to", market_vars.index.max())
print("Shape:", market_vars.shape)

Saved Step 02 outputs.
Date range: 2007-07-31 00:00:00 to 2026-06-04 00:00:00
Shape: (4685, 5)
